<div style="
  background: linear-gradient(145deg, #1a0b08, #2d1310);
  border: 4px solid transparent;
  border-radius: 14px;
  padding: 18px 22px;
  margin: 12px 0;
  font-size: 26px;
  font-weight: 600;
  color: #fff8f6;
  box-shadow: 0 6px 14px rgba(0,0,0,0.3);
  background-clip: padding-box;
  position: relative;
">
  <div style="
    position: absolute;
    inset: 0;
    padding: 4px;
    border-radius: 14px;
    background: linear-gradient(90deg, #ff7b00, #ff0054, #9d0208);
    -webkit-mask: 
      linear-gradient(#fff 0 0) content-box, 
      linear-gradient(#fff 0 0);
    -webkit-mask-composite: xor;
    mask-composite: exclude;
    pointer-events: none;
  "></div>
  
  <b>03 $\rightarrow$ Application Evaluation Workflow</b>
  <br>
  <span style="color:#ffb5a7; font-size: 18px;">(Architecture, Implementation, and Continuous Optimization)</span>
</div>

---

# Table of Contents

1. [Introduction to the LLM Application Evaluation Lifecycle](#1-introduction-to-the-llm-application-evaluation-lifecycle)
   - 1.1 [Systemic Overview](#11-systemic-overview)
   - 1.2 [The Closed-Loop Continuous Evaluation Paradigm](#12-the-closed-loop-continuous-evaluation-paradigm)
   - 1.3 [Multi-Pipeline Evaluation Architectures](#13-multi-pipeline-evaluation-architectures)

2. [Learning Objectives](#2-learning-objectives)
3. [Topic 1: The Step-by-Step Evaluation Workflow](#3-topic-1-the-step-by-step-evaluation-workflow)
   - 3.1 [Overview](#31-overview)
   - 3.2 [Step 1: Define Task & Target System](#32-step-1-define-task--target-system)
   - 3.3 [Step 2: Define Success Criteria & Metrics](#33-step-2-define-success-criteria--metrics)
   - 3.4 [Step 3: Curate Golden Dataset](#34-step-3-curate-golden-dataset)
   - 3.5 [Step 4: Select Evaluation Method](#35-step-4-select-evaluation-method)
   - 3.6 [Step 5: Run Model & Evaluate Metrics](#36-step-5-run-model--evaluate-metrics)
   - 3.7 [Step 6: Error Analysis & Diagnostics](#37-step-6-error-analysis--diagnostics)
   - 3.8 [Step 7: Iterative Hardening & System Optimization](#38-step-7-iterative-hardening--system-optimization)
   - 3.9 [Step 8: Production Deployment & Online Monitoring](#39-step-8-production-deployment--online-monitoring)
   - 3.10 [Step 9: Production Failure Feedback Loop](#310-step-9-production-failure-feedback-loop)
   - 3.11 [Best Practices & Common Mistakes](#311-best-practices--common-mistakes)
   - 3.12 [Key Takeaways](#312-key-takeaways)

4. [Topic 2: Real-World Implementation – Automated Email Classifier Evaluation Pipeline](#4-topic-2-real-world-implementation--automated-email-classifier-evaluation-pipeline)
   - 4.1 [Overview](#41-overview)
   - 4.2 [Architecture & Data Pipeline Design](#42-architecture--data-pipeline-design)
   - 4.3 [Code Implementation](#43-code-implementation)
   - 4.4 [Code Walkthrough & Expected Output](#44-code-walkthrough--expected-output)
   - 4.5 [Iterative Versioning: Model and Prompt Hardening](#45-iterative-versioning-model-and-prompt-hardening)
   - 4.6 [Best Practices & Common Mistakes](#46-best-practices--common-mistakes)
   - 4.7 [Key Takeaways](#47-key-takeaways)

---


<a id="1-introduction-to-the-llm-application-evaluation-lifecycle"></a>
##

<span style="
  display: inline-block;
  color: #fff;
  background: linear-gradient(135deg, #7209b7, #4cc9f0);
  padding: 12px 20px;
  border-radius: 12px;
  font-size: 24px;
  font-weight: 700;
  box-shadow: 0 4px 12px rgba(0,0,0,0.3);
  transition: transform 0.2s ease, box-shadow 0.2s ease;
">1. Introduction to the LLM Application Evaluation Lifecycle</span>

<a id="11-systemic-overview"></a>
### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">1.1 Systemic Overview</span>

Evaluating modern LLM applications requires looking beyond raw base-model benchmark scores to analyze the entire application pipeline as an integrated system.

#### Key Pillars of Systemic Evaluation:

- **Full-Stack Application Boundaries**
    - **Retrievers** — fetch relevant document chunks from vector stores; failures include returning irrelevant context or missing the target document entirely.
    - **Re-rankers** — re-score and reorder retrieved chunks by query relevance; failures cause critical context to be buried at low-attention positions.
    - **System prompts** — define behavioral constraints and output format; poorly crafted prompts produce inconsistent or off-topic responses.
    - **Guardrails** — enforce safety constraints on inputs and outputs; bypassed guardrails expose users to harmful content.
    - **Structured output parsers** — extract JSON, function calls, or structured data from raw LLM text; malformed parsing breaks downstream tool execution.
    - **Model generation** — the core LLM inference step; inherent hallucination tendencies and knowledge cutoffs are baseline failure modes.

- **Multi-Dimensional Metrics**
    - **Functional accuracy** — does the output correctly answer the user's question?
    - **Factual groundedness** — is every claim supported by the retrieved context, or did the model introduce unsupported information?
    - **Safety** — is the output free from toxicity, PII leakage, and adversarial bypass vulnerabilities?
    - **Latency (TTFT)** — Time To First Token measures initial responsiveness; high TTFT degrades perceived performance.
    - **Operational token costs** — every API call consumes tokens; unoptimized pipelines can drive costs exponentially at scale.

- **End-to-End Lifecycle Alignment**
    - Transitioning seamlessly from development-phase offline regression suites to continuous production monitoring.
    - Offline evaluations gate deployments; online evaluations detect drift and emergent failures in live traffic.


<a id="12-the-closed-loop-continuous-evaluation-paradigm"></a>
### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">1.2 The Closed-Loop Continuous Evaluation Paradigm</span>

Production-grade AI systems rely on a closed-loop feedback paradigm that connects offline testing with live production telemetry.

#### Closed-Loop Lifecycle Stages:

* **Offline Evaluation Gate**
  * Validating candidate system prompts, model versions, and retrieval parameters against a version-controlled Golden Dataset before deployment.
  * Acts as a binary CI/CD gate — configurations that fail to meet threshold scores are automatically blocked from production.

* **Live Telemetry & Production Monitoring**
  * Tracking user feedback (thumbs up/down), support escalations, session abandonments, and latency metrics on live production traffic.
  * Computed signals (faithfulness, toxicity) are evaluated asynchronously on sampled traffic to avoid impacting user response times.

* **Dataset Augmentation**
  * Capturing real-world failure edge cases and automatically feeding them back into the offline Golden Dataset for continuous regression testing.
  * Ensures that every production failure encountered is permanently guarded against in future evaluation cycles.


<img src="../assets/nb_assets/nb0302.png" alt="nb0302.png" style="width:100%; max-width:800px; display:block; margin:auto;" />

<a id="13-multi-pipeline-evaluation-architectures"></a>
### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">1.3 Multi-Pipeline Evaluation Architectures</span>

Sub-system components require dedicated evaluation pipelines tailored to their specific operational function:

| Pipeline | Measurement Focus |
| --- | --- |
| **1. Retriever Evaluation Pipeline** | Measures Context Recall & Precision. |
| **2. Re-Ranker Evaluation Pipeline** | Measures Mean Reciprocal Rank (MRR). |
| **3. Generation Evaluation Pipeline** | Measures Faithfulness & Answer Relevance. |
| **4. Operational Pipeline** | Measures Latency, Time-To-First-Token (TTFT), and Token Cost. |
| **5. Security Evaluation Pipeline** | Measures Resistance to Prompt Injection. |

<a id="2-learning-objectives"></a>
##

<span style="
  display: inline-block;
  color: #fff;
  background: linear-gradient(135deg, #7209b7, #4cc9f0);
  padding: 12px 20px;
  border-radius: 12px;
  font-size: 24px;
  font-weight: 700;
  box-shadow: 0 4px 12px rgba(0,0,0,0.3);
  transition: transform 0.2s ease, box-shadow 0.2s ease;
">
  2. Learning Objectives
</span>

1. **Design and Execute** a structured, end-to-end evaluation lifecycle for any production LLM application.
2. **Curate and Manage** version-controlled **Golden Datasets** tailored to domain-specific business goals.
3. **Select and Implement** appropriate evaluation methods (Deterministic Code Assertions, Human-in-the-Loop, or LLM-as-a-Judge) based on output complexity.
4. **Build** automated evaluation pipelines in Python using Pydantic, computing statistical metrics across iterations.
5. **Architect** production feedback loops that extract real-world deployment failures and integrate them back into offline regression suites.

<a id="3-topic-1-the-step-by-step-evaluation-workflow"></a>
##

<span style="
  display: inline-block;
  color: #fff;
  background: linear-gradient(135deg, #7209b7, #4cc9f0);
  padding: 12px 20px;
  border-radius: 12px;
  font-size: 24px;
  font-weight: 700;
  box-shadow: 0 4px 12px rgba(0,0,0,0.3);
  transition: transform 0.2s ease, box-shadow 0.2s ease;
">
  3. Topic 1: The Step-by-Step Evaluation Workflow
</span>

<a id="31-overview"></a>
### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">3.1 Overview</span>

> **Goal**: Replaces informal `vibe testing` with reproducible, quantitative quality release gates.

<img src="../assets/nb_assets/nb0301.jpg" alt="nb0301.jpg" style="width:100%; max-width:500px; display:block; margin:auto;" />

<a id="32-step-1-define-task--target-system"></a>
### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">3.2 Step 1: Define Task & Target System</span>


* **Target System Scope**
  * Define whether evaluating an isolated classifier, RAG pipeline, or multi-step autonomous agent.
  * The evaluation strategy, metrics, and dataset design change dramatically based on the system architecture under test.

* **Task Domain**
  * Identify output category (e.g., text classification, structured JSON extraction, semantic summarization, code generation).
  * Each domain has different ground-truth requirements and scoring methodologies.

* **Task-to-Metric Mapping**
  * Align each task domain with appropriate evaluation metrics.
  * Misaligned metrics produce misleading scores — e.g., using exact-match accuracy on open-ended summarization tasks.


#### Task-To-Metric Mapping Architecture

| Task Type | Target Metrics |
| --- | --- |
| **Text Classification** | Classification Accuracy, Precision, Recall, F1 |
| **RAG Document Retrieval** | Context Recall, Context Precision, MRR |
| **RAG Output Generation** | Faithfulness / Groundedness, Answer Relevance |
| **Structured Extraction** | Schema Adherence Rate, Field-Level Exact Match |
| **Operational Constraints** | Latency (ms), Time-To-First-Token (TTFT), Cost ($) |

<a id="33-step-2-define-success-criteria--metrics"></a>
### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">3.3 Step 2: Define Success Criteria & Metrics</span>

* **Quantitative Release Gates**
  * Set explicit pass/fail thresholds (e.g., >= 95% classification accuracy, >= 0.90 groundedness score, latency < 800ms).
  * These thresholds serve as binary CI/CD gates — failing any threshold blocks deployment.

* **Domain Metric Alignment**
  * *Classification / Routing*: Accuracy, Precision, Recall, Macro/Micro F1-Score.
  * *RAG Systems*: Faithfulness/Groundedness, Context Recall, Context Precision, Answer Relevance.
  * *Operational Performance*: Time-To-First-Token (TTFT), Total Latency (ms), Token Cost ($).

* **Baseline Comparison**
  * Set reference benchmarks against existing heuristic rule engines, human evaluator baselines, or prior model versions.
  * Without a baseline, you cannot determine whether a new configuration represents improvement or regression.


<a id="34-step-3-curate-golden-dataset"></a>
### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">3.4 Step 3: Curate Golden Dataset</span>

* **Ground-Truth Benchmark**
  * Curate 50 to 500 verified ground-truth rows for offline development testing (expanding to thousands in production).
  * Each row must be independently verified by domain experts to ensure label accuracy.

* **Schema Uniformity**
  * Standardize row fields across `input_prompt`, `retrieved_context`, `ground_truth_target`, and `metadata`.
  * Consistent schema enables automated batch processing and cross-version comparison.

* **Edge-Case Coverage**
  * Include noisy, ambiguous, incomplete, and adversarial user queries alongside typical production prompts.
  * Edge cases are statistically certain to appear at production scale — omitting them creates blind spots in the evaluation.

<br>

```python
GoldenDatasetSchema = {
    "id": str,
    "input_text": str,
    "retrieved_context": Optional[List[str]],
    "ground_truth_target": str,
    "metadata": Dict[str, Any]
}
```


<img src="../assets/nb_assets/nb0303.jpg" alt="nb0303.jpg" style="width:100%; max-width:700px; display:block; margin:auto;" />

In [1]:
# Golden Dataset Manager
import json

class GoldenDataset:
    def __init__(self, name):
        self.name = name
        self.records = []

    def add_record(self, input_text, expected, category):
        record = {
            "id": f"GD-{len(self.records)+1:03d}",
            "input": input_text,
            "expected": expected,
            "category": category,
        }
        self.records.append(record)

    def summary(self):
        cats = {}
        for r in self.records:
            cats[r["category"]] = cats.get(r["category"], 0) + 1
        return {"total": len(self.records), "categories": cats}

gd = GoldenDataset("Customer Support Classifier Dataset")
gd.add_record("Double charged on my card", "Billing", "Billing")
gd.add_record("App crashes on startup", "Technical", "Technical")
gd.add_record("What are business hours?", "General", "General")

print("=" * 60)
print(f"Golden Dataset: {gd.name}")
print("=" * 60)
print(json.dumps(gd.summary(), indent=2))

Golden Dataset: Customer Support Classifier Dataset
{
  "total": 3,
  "categories": {
    "Billing": 1,
    "Technical": 1,
    "General": 1
  }
}


<a id="35-step-4-select-evaluation-method"></a>
### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">3.5 Step 4: Select Evaluation Method</span>

* **Selection Criteria**
  * Balance output complexity, execution speed requirements, reproducibility, and token budget constraints.
  * The optimal method depends on whether outputs are structured (JSON, categories) or open-ended (summaries, creative text).

* **Evaluation Methodologies**
  * **Deterministic Code Assertions** — ultra-fast (<10ms), 100% reproducible, zero token cost for JSON schemas and categorical routing.
    * Ideal for: exact-match classification, schema validation, regex pattern checks.
  * **LLM-as-a-Judge** — scalable, flexible evaluation (1-3s) for open-ended text quality, RAG faithfulness, and tone alignment.
    * Ideal for: summarization quality, answer helpfulness, groundedness verification.
  * **Human-in-the-Loop (HITL)** — high-precision qualitative audit for high-risk domains (medical, legal, finance) and initial gold-standard labeling.
    * Ideal for: establishing ground-truth baselines, validating LLM judge alignment, red-teaming safety.

#### Evaluation Methodology Comparison Matrix:

| Evaluation Method | Ideal Use Case | Key Advantages | Primary Drawbacks | Execution Speed |
| :--- | :--- | :--- | :--- | :--- |
| **Deterministic Code Assertions** | Categorical routing, structured JSON parsing, exact match string comparisons. | Ultra fast, 100% reproducible, zero token cost. | Inflexible for creative, open-ended text outputs. | Ultra Fast (<10ms) |
| **LLM-as-a-Judge** | Open-ended text quality, RAG faithfulness, summarization accuracy, tone audit. | Scalable, aligns closely with human preference when prompted properly. | Non-zero token cost, potential judge bias, mild variance. | Moderate (1-3s) |
| **Human-in-the-Loop (HITL)** | High-risk domain audits (legal, medical, finance), initial gold-standard annotation. | Maximum domain accuracy and qualitative nuance. | Slow, expensive, unscalable for continuous CI/CD pipelines. | Manual / Slow |


<a id="36-step-5-run-model--evaluate-metrics"></a>
### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">3.6 Step 5: Run Model & Evaluate Metrics</span>

* **Batch Execution**
  * Execute target system over the entire Golden Dataset with zero temperature ($T=0.0$) for deterministic reproducibility.
  * Batch processing ensures every test case is evaluated under identical conditions.

* **Metric Computation**
  * Compare model predictions against verified reference ground truth to calculate aggregate accuracy, recall, and distance metrics.
  * Compute per-category breakdowns to identify systematic weaknesses in specific query types.

* **Diagnostic Logging**
  * Record input prompts, model outputs, target labels, pass/fail status, and execution metadata.
  * Detailed logs enable rapid root-cause analysis when evaluation scores drop below thresholds.


<a id="37-step-6-error-analysis--diagnostics"></a>
### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">3.7 Step 6: Error Analysis & Diagnostics</span>

* **Taxonomy Classification**
  * Categorize failed samples into specific error taxonomies (e.g., boundary ambiguity, hallucinations, context omission).
  * Systematic categorization reveals whether failures are concentrated in specific query types or spread uniformly.

* **Diagnostic Trace Inspection**
  * Review intermediate reasoning steps, prompt instructions, and context chunks to isolate root causes.
  * Trace inspection distinguishes between retrieval failures (wrong context fetched) and generation failures (correct context ignored).


<a id="38-step-7-iterative-hardening--system-optimization"></a>
### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">3.8 Step 7: Iterative Hardening & System Optimization</span>

* **System Prompt Refinement**
  * Add boundary conditions, explicit formatting constraints, or targeted few-shot examples to resolve ambiguity.
  * Version-control every prompt iteration to enable precise A/B comparisons against the Golden Dataset.

* **Model & Retrieval Upgrades**
  * Upgrade to higher-capacity models or adjust RAG chunk sizes, embeddings, re-rankers, and top-$k$ parameters.
  * Test one variable at a time — changing multiple parameters simultaneously makes it impossible to attribute improvements.

* **Regression Re-Testing**
  * Re-evaluate updated system configurations against the Golden Dataset until performance satisfies target release gates.
  * Verify that fixing failures in one category does not introduce regressions in previously passing categories.


<img src="../assets/nb_assets/nb0304.jpg" alt="nb0304.jpg" style="width:100%; max-width:650px; display:block; margin:auto;" />

<a id="39-step-8-production-deployment--online-monitoring"></a>
### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">3.9 Step 8: Production Deployment & Online Monitoring</span>


* **Deployment Gate Approval**
  * Release application to production only after passing offline Golden Dataset evaluation gates.
  * Gate approval requires simultaneous pass across all metric dimensions — accuracy, safety, and latency.

* **Operational Telemetry**
  * Monitor real-time Time-To-First-Token (TTFT), total end-to-end latency, throughput, and token cost ($).
  * Set up automated alerting for latency spikes, error rate increases, and cost threshold breaches.

* **Quality Sampling**
  * Continuously sample live production traffic for ongoing automated LLM-as-a-Judge or guardrail audits.
  * Use stratified sampling to prioritize high-risk segments (negative feedback, escalations) over normal traffic.


<a id="310-step-9-production-failure-feedback-loop"></a>
### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">3.10 Step 9: Production Failure Feedback Loop</span>

* **Failure Capture**
  * Extract negative user feedback, support escalations, and guardrail violations from production telemetry.
  * Automatically tag and categorize production failures by error taxonomy for targeted remediation.

* **Dataset Augmentation**
  * Annotate production failure cases with verified ground-truth targets and append them to the offline Golden Dataset to prevent regression recurrence.
  * This creates a continuously growing test suite that reflects the true distribution of real-world edge cases.


<a id="311-best-practices--common-mistakes"></a>
### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">3.11 Best Practices & Common Mistakes</span>

#### Best Practices

* **Version-Control Golden Datasets**: Store evaluation datasets alongside code repositories, tracking schema and target label updates.
* **Automate CI/CD Release Gates**: Run automated evaluation suites on pull requests before merging changes to main branches.
* **Fix Temperature to Zero ($T=0.0$)**: Fix $T=0.0$ during evaluation runs to ensure consistent, reproducible results.

#### Common Mistakes

* **Evaluating on Synthetic Data Alone**: Relying exclusively on synthetically generated test prompts while ignoring real user query distributions.
* **Ignoring Misclassification Patterns**: Reviewing top-line accuracy while ignoring systematic failures on specific query categories.
* **Neglecting Production Feedback Loops**: Deploying without mechanisms to capture and learn from live production failure edge cases.

<a id="312-key-takeaways"></a>
### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">3.12 Key Takeaways</span>

* **Sequential 9-Step Lifecycle**: Provides a systematic blueprint from task definition to continuous production failure feedback.
* **Closed-Loop Reliability**: Offline Golden Dataset gates and live production telemetry work together to ensure long-term model safety and accuracy.

<a id="4-topic-2-real-world-implementation--automated-email-classifier-evaluation-pipeline"></a>
##

<span style="
  display: inline-block;
  color: #fff;
  background: linear-gradient(135deg, #7209b7, #4cc9f0);
  padding: 12px 20px;
  border-radius: 12px;
  font-size: 24px;
  font-weight: 700;
  box-shadow: 0 4px 12px rgba(0,0,0,0.3);
  transition: transform 0.2s ease, box-shadow 0.2s ease;
">
  4. Topic 2: Real-World Implementation – Automated Email Classifier Evaluation Pipeline
</span>

<a id="41-overview"></a>
### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">4.1 Overview</span>

* **System Under Test**: Enterprise customer support router analyzing incoming customer emails.
* **Operational Routing Categories**:
  * `Billing`: Routes financial, invoice, and payment queries to financial support.
  * `Technical`: Routes app crashes, software bugs, and API errors to technical support.
  * `General`: Routes business hours, policy, and general questions to standard customer service.

<a id="42-architecture--data-pipeline-design"></a>
### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">4.2 Architecture & Data Pipeline Design</span>

* **Pipeline Sequence**: Incoming Customer Email $\rightarrow$ Pydantic Schema Validation $\rightarrow$ Zero-Temperature Inference ($T=0.0$) $\rightarrow$ Ground-Truth Label Matcher $\rightarrow$ Release Gate Evaluator.
* **Contract Enforcement**: Pydantic models guarantee explicit category bounds and structure diagnostic report outputs.

<img src="../assets/nb_assets/nb0305.jpg" alt="nb0305.jpg" style="width:100%; max-width:600px; display:block; margin:auto;" />

<a id="43-code-implementation"></a>
### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">4.3 Code Implementation</span>

* **Implementation Goal**: Build a deterministic Python evaluation harness for email routing with automated release threshold gates.
* **Prerequisites & Dependencies**:

```bash
pip install openai pydantic
```

In [ ]:
# End-to-End Email Classifier Evaluation Suite
GOLDEN_DATASET = [
    {"id": "E-01", "text": "I was double charged on my card", "label": "Billing"},
    {"id": "E-02", "text": "The mobile app crashes on login", "label": "Technical"},
    {"id": "E-03", "text": "What are your support hours?", "label": "General"},
    {"id": "E-04", "text": "Please issue a refund for order #12", "label": "Billing"},
    {"id": "E-05", "text": "API returns server error 500", "label": "Technical"},
]

def classify_email(text):
    t = text.lower()
    if "charged" in t or "refund" in t or "billing" in t:
        return "Billing"
    if "crashes" in t or "error" in t or "api" in t:
        return "Technical"
    return "General"

print("=" * 60)
print("EVALUATION RUN: Email Classifier")
print("=" * 60)

passed = 0
for sample in GOLDEN_DATASET:
    pred = classify_email(sample["text"])
    correct = pred == sample["label"]
    passed += correct
    status = "[PASS]" if correct else "[FAIL]"
    print(f"{status} [{sample['id']}] Expected: {sample['label']:<10} | Pred: {pred:<10} | Text: '{sample['text']}'")

acc = (passed / len(GOLDEN_DATASET)) * 100
print(f"\nAccuracy: {passed}/{len(GOLDEN_DATASET)} ({acc:.1f}%)")

EVALUATION RUN: Email Classifier
[PASS] [E-01] Expected: Billing    | Pred: Billing    | Text: 'I was double charged on my card'
[PASS] [E-02] Expected: Technical  | Pred: Technical  | Text: 'The mobile app crashes on login'
[PASS] [E-03] Expected: General    | Pred: General    | Text: 'What are your support hours?'
[PASS] [E-04] Expected: Billing    | Pred: Billing    | Text: 'Please issue a refund for order #12'
[PASS] [E-05] Expected: Technical  | Pred: Technical  | Text: 'API returns server error 500'

Accuracy: 5/5 (100.0%)


In [ ]:
# Evaluation Report Generator with Automated Release Gates
import json

def generate_report(results, min_accuracy_threshold=80.0):
    total = len(results)
    correct = sum(1 for r in results if r["correct"])
    accuracy = (correct / total) * 100 if total > 0 else 0.0
    
    passed_gate = accuracy >= min_accuracy_threshold
    
    report = {
        "summary": {
            "total_samples": total,
            "correct_predictions": correct,
            "accuracy_percent": round(accuracy, 2),
            "release_threshold": min_accuracy_threshold,
        },
        "gate_status": "APPROVED" if passed_gate else "BLOCKED",
        "failed_samples": [r for r in results if not r["correct"]],
    }
    return report

eval_data = [
    {"id": "1", "correct": True},
    {"id": "2", "correct": True},
    {"id": "3", "correct": True},
    {"id": "4", "correct": False, "reason": "Confused billing query with technical error"},
]

rep = generate_report(eval_data, min_accuracy_threshold=80.0)
print("=" * 60)
print("AUTOMATED RELEASE GATE REPORT")
print("=" * 60)
print(json.dumps(rep, indent=2))

AUTOMATED RELEASE GATE REPORT
{
  "summary": {
    "total_samples": 4,
    "correct_predictions": 3,
    "accuracy_percent": 75.0,
    "release_threshold": 80.0
  },
  "gate_status": "BLOCKED",
  "failed_samples": [
    {
      "id": "4",
      "correct": false,
      "reason": "Confused billing query with technical error"
    }
  ]
}


<a id="44-code-walkthrough--expected-output"></a>
### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">4.4 Code Walkthrough & Expected Output</span>

#### Code Walkthrough

1. **Data Schemas**: Enforces explicit category constraints (`Billing`, `Technical`, `General`) and structures diagnostic results using Pydantic.
2. **Golden Dataset Setup**: Curates representative ground-truth email test samples with verified target labels.
3. **Classifier Engine**: Executes zero-temperature structured classification.
4. **Release Gate Audit**: Calculates accuracy against pass thresholds (e.g., 80%) and flags failed samples.

#### Expected Diagnostic Output

```text
============================================================
AUTOMATED RELEASE GATE REPORT
============================================================
{
  "summary": {
    "total_samples": 4,
    "correct_predictions": 3,
    "accuracy_percent": 75.0,
    "release_threshold": 80.0
  },
  "gate_status": "BLOCKED",
  "failed_samples": [
    {
      "id": "4",
      "correct": false,
      "reason": "Confused billing query with technical error"
    }
  ]
}
```

<a id="45-iterative-versioning-model-and-prompt-hardening"></a>
### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">4.5 Iterative Versioning: Model and Prompt Hardening</span>

* **Hardening Workflow**: When accuracy falls below release gates (e.g., achieving 75% when 80% is required), developers perform prompt and model hardening cycles.

When evaluation scores fall below targeted release thresholds (e.g., achieving 80% accuracy when 95% is required), developers iterate through system hardening cycles:

<img src="../assets/nb_assets/nb0306.jpg" alt="nb0306.jpg" style="width:100%; max-width:800px; display:block; margin:auto;" />

1. **System Prompt Iteration**: Upgrade from Prompt $V_1$ to Prompt $V_2$ by adding explicit category boundary definitions to resolve ambiguity.
2. **Regression Testing**: Re-test updated configuration ($V_2$) against the same Golden Dataset to confirm improvement without introducing regressions.
3. **Release Approval**: Approve configuration for production deployment once accuracy satisfies release threshold gates.

<a id="46-best-practices--common-mistakes"></a>
### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">4.6 Best Practices & Common Mistakes</span>

#### Best Practices

* **Zero Temperature for Testing**: Fix `temperature=0.0` during evaluation runs to ensure consistent, reproducible outputs.
* **Log Diagnostic Reasoning**: Capture intermediate reasoning steps alongside predictions for fast root-cause analysis.
* **Typed Contract Validation**: Use Pydantic models to validate structured outputs programmatically.

#### Common Mistakes

* **Testing on Prompt Few-Shot Examples**: Including the exact same examples in the Golden Dataset that were used in prompt demonstrations.
* **Ignoring Category-Specific Failure Patterns**: Reviewing overall accuracy scores while ignoring systematic errors on specific query categories.

<a id="47-key-takeaways"></a>
### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">4.7 Key Takeaways</span>

* **Objective Automation**: Automated Python evaluation pipelines allow developers to measure system performance deterministically across code and prompt iterations.
* **Structured Reliability**: Structured data parsing (using Pydantic) enables automated comparison between predicted outputs and reference ground-truth targets.